<a href="https://colab.research.google.com/github/JChengecha/medallion-pipeline/blob/main/project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 1: Medallion Pipeline
This notebook builds the medallion pipeline one step at a time.

- **Bronze** - raw data, exactly as received. No cleaning here.
- **Silver** - cleaned and deduplicated, using MERGE.
- **Gold** - business-ready aggregates.

<DETAILS>
<SUMMARY>IN THIS NOTEBOOK</SUMMARY>

* **Setup & Config**: Spark Session tuned for Delta Lake.  
* **Bronze Layer**: Automated download & raw Delta landing.  
* **Silver Layer**: Data cleansing, deduplication, surrogate key generation (trip_id), and an idempotent Delta MERGE.  
* **Gold Layer**: Business aggregation for reporting/dashboards.    
* **Audit / Quality Gate**: Automated data and financial reconciliation.  
* Data -> [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
</DETAILS>

## Step 0 - setup
- create a spark session with delta extaension and define ouptut paths

In [1]:
import os
%pip install -q pyspark delta-spark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

spark = (configure_spark_with_delta_pip(
    SparkSession.builder.master("local[2]")
    .appName("Project_1")
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions","4")
    .config("spark.ui.showConsoleProgress","false")
)).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

BRONZE = os.path.expanduser("~/projects/data-engineering/Project_1/proj1_bronze")
SILVER = os.path.expanduser("~/projects/data-engineering/Project_1/proj1_silver")
GOLD = os.path.expanduser("~/projects/data-engineering/Project_1/proj1_gold")

- Spin up a Spark engine configured to work with Delta Lake tables.
- Read raw files using an enforced schema (so errors fail fast).
- Transform the columns using Spark functions (F).

## Step 1: Bronze (raw data as received)
write the raw orders table into bronze Delta table

In [2]:
import urllib.request

#1. define the url and bronze landing path
year, month = "2025","01"
url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month}.parquet"

raw_landing_folder = os.path.join(BRONZE,"raw_landing")
os.makedirs(raw_landing_folder,exist_ok=True)

local_parquet_path = os.path.join(raw_landing_folder,f"yellow_{year}_{month}.parquet")
delta_bronze_table = os.path.join(BRONZE,"yellow_taxi_delta")

#2. download,  only if  not on disk, the parquet file from the url and save it to the bronze landing path
if not os.path.exists(local_parquet_path):
    print(f"Downloading {url} to {local_parquet_path}...")
    urllib.request.urlretrieve(url, local_parquet_path)
else:
    print(f"File {local_parquet_path} already exists. Skipping download.")

#3. read the parquet file into a Spark DataFrame
src = os.path.basename(local_parquet_path) #eg. yellow_2025_01.parquet

df = (
    spark.read.parquet(local_parquet_path)
    .withColumn("_source_file",F.lit(src))
    .withColumn("_ingest_at",F.current_timestamp())
    )

#4. INDEMPOTENT WRITE: replace only rows for this source file
(
    df.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere",f"_source_file = '{src}'")
    .option("overwriteSchema","true")
    .save(delta_bronze_table)
)

print(f"Ingested {df.count():,} rows into the Bronze Delta for {src}")

Ingested 3,475,226 rows into the Bronze Delta for yellow_2025_01.parquet


**pipeline code fetches the file dynamically**, lands it in your BRONZE zone as raw data, and Spark immediately picks it up:

<details>
<summary>Method 2: The Zero-Disk Way (Stream over HTTP directly into Memory)</summary>
  
If you have strict disk space limits and want to read the URL directly over the internet without saving any intermediate `.parquet` file to your drive, you can use **DuckDB** with **PyArrow** as a bridge.


DuckDB can read remote HTTP Parquet files on the fly using range requests:
```
pip install duckdb pyarrow
import duckdb

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

# 1. Read remote HTTPS directly into an in-memory Apache Arrow table
arrow_table = duckdb.query(f"SELECT * FROM '{url}'").arrow()

# 2. Turn the Arrow table into a Spark DataFrame
df = spark.createDataFrame(arrow_table.to_pandas())

# 3. Write directly to your Bronze Delta Lake table
delta_bronze_table = os.path.join(BRONZE, "yellow_taxi_delta")
df.write.format("delta").mode("append").save(delta_bronze_table)
```
**How it works:** DuckDB pulls only the needed chunks from CloudFront over HTTPS into RAM, converts them to Arrow, hands them to Spark, and Spark writes them directly into your local Delta Lake directory—bypassing the manual download step entirely.
</details>

In [3]:
from IPython.display import IFrame, display

display(
    IFrame(
        "https://docs.google.com/gview?embedded=true&url=https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf",
        width="100%",
        height=500
    )
    )

## Step 2: Silver (Clean and deduplicate with merge)
- reads bronze, cleans it(dedup by order_id), and merges the cleaned rows into silver.  
- reruns must be safe. MERGE is indempotent, a blind append would double the data.

In [4]:

from pyspark.sql.window import Window as window


#1. Define paths matching Bronze ingestion step
delta_bronze_table = os.path.join(BRONZE, "yellow_taxi_delta")
delta_silver_table = os.path.join(SILVER, "yellow_taxi_delta")
delta_quarantine_table = os.path.join(SILVER,"yellow_taxi_quarantine_delta")

#2. Read BRONZE Delta table
bronze_df = spark.read.format("delta").load(delta_bronze_table)

#3. clean  and Enrich for Silver
#   a. create a unique "trip_id" by hashing identifying columns
#   b. Tag invalid records with an explicit failure reason instead of discarding them, and write them to a quarantine table for later inspection

#define a windowpartitioned by trip_id
window_spec = window.partitionBy("trip_id").orderBy("tpep_pickup_datetime")

tagged_df = (
  bronze_df.withColumn("trip_id",
    F.sha2(F.concat_ws("||",
      F.coalesce(F.col("VendorID").cast("string"),F.lit("")),
      F.coalesce(F.col("tpep_pickup_datetime").cast("string"),F.lit("")),
      F.coalesce(F.col("tpep_dropoff_datetime").cast("string"),F.lit("")),
      F.coalesce(F.col("PULocationID").cast("string"),F.lit("")),
      F.coalesce(F.col("DOLocationID").cast("string"),F.lit("")),
      F.coalesce(F.col("fare_amount").cast("string"),F.lit(""))
    ),256)
  )
  #Assign row number within duplicates(1 = primary copy, >1 = duplicate copy)
  .withColumn("duplicate_rank",F.row_number().over(window_spec))
  .withColumn("quarantine_reason",
               F.when(F.col("duplicate_rank") > 1, "Duplicate record") # <----- catches the duplciates
               .when(F.col("tpep_pickup_datetime").isNull(), "Missing pickup datetime")
               .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"), "Invalid trip duration")
               .when(F.col("trip_distance") < 0, "Negative trip distance")
               .when(F.col("fare_amount") < 0, "Negative fare amount")
               .when(
                   (F.year("tpep_pickup_datetime") != int(year)) |
                    (F.month("tpep_pickup_datetime") != int(month)),
                    "Pickup datetime not in specified year/month")
               .otherwise(None)
               ).drop("duplicate_rank"))


#4. split stream into valid(silver) vs. invalid(quarantine) records
valid_df = (
    tagged_df.filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason")
    .cache() #keeps records in memory
)

quarantine_df = (
  tagged_df.filter(F.col("quarantine_reason").isNotNull())
)


#5. check if Silver Table exists and has data
is_first_load = not DeltaTable.isDeltaTable(spark,delta_silver_table) or(
    spark.read.format("delta").load(delta_silver_table).limit(1).count()==0)

if is_first_load:
  # FASTPATH: direct append on first load which bypassess expensive full join
  print("Silver table empty. Performing fast initial write...")
  valid_df.write.format("delta").mode("overwrite").save(delta_silver_table)
else:
  #Incremental path: merge updates/new rows
  print("Existing data detected; Performing Delta MERFE...")
  target_silver = DeltaTable.forPath(spark,delta_silver_table)
  target_silver.alias("target")\
    .merge(valid_df.alias("source"), "target.trip_id = source.trip_id")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

#6. Append Quarantined records directly
print("Writing quarantined records...")
quarantine_df.write.format("delta").mode("append").save(delta_quarantine_table)

#7. Unpersist cache to free RAM
valid_df.unpersist()

#8.  report
silver_count = spark.read.format("delta").load(delta_silver_table).count()
quarantine_count = spark.read.format("delta").load(delta_quarantine_table).count()

print(
   f"{'-'*5} INGESTION REPORT {'-'*5}\n"
   f"Rows in Silver (CLEAN): {silver_count:,}\n"
   f"Rows in Quarantine: {quarantine_count:,}"
)

Silver table empty. Performing fast initial write...
Writing quarantined records...
----- INGESTION REPORT -----
Rows in Silver (CLEAN): 3,330,662
Rows in Quarantine: 144,564


##### OBSERVATIONS
<details><summary>writing on local vs cloud</summary>
`import shutil
shutil.rmtree(delta_silver_table, ignore_errors=True)`

**Why**: running locally on your machine or Colab disk (`/root/proj1_silver`). Local Python file operations work normally here.When you deploy this to the cloud (AWS S3 or Azure ADLS via s3://... or abfss://...), shutil.rmtree will fail because Python cannot treat cloud object stores like local hard drive folders

 - In cloud environments, you let Delta overwrite schemas natively:
```p
cleaned.limit(0).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_silver_table)
```
</details>

---
<details><summary>performance</summary>
When running `.merge()`, Spark will evaluate the cleaned dataframe twice **(once to find matching keys**, and **once to read the payload to insert/update**).   
If you notice your Colab cell taking a long time, cache the cleaned records in memory before the merge:

```
# Cache before the merge to prevent Spark from recomputing hashes and filters
cleaned = cleaned.cache()

# Execute merge
target_table.alias("target") \
  ...
  .execute()

# Unpersist after merge is complete
cleaned.unpersist()
```
</details>

---
<details><summary>adding to Silver</summary>
Silver is also where you enrich data with helpful derived columns. Adding these saves time when building the Gold layer:      

- `trip_duration_minutes`

```  
(F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60
```   
-  `pickup_date`  

```
F.to_date(F.col("tpep_pickup_datetime"))
```
This date column can also be used as a partition column to make your Gold layer aggregations much faster
</details>

---

<DETAILS><SUMMARY>OTHER</SUMMARY>

* **No Single `order_id`**: NYC TLC records do not come with an order ID. To merge and deduplicate, data engineers create a surrogate trip_id by hashing key columns (VendorID, pickup time, dropoff time, and location IDs).

* **Quality Cleaning (The Silver Standard)**: Real Silver tables filter out bad data—like trips where the dropoff is before the pickup, or trips with negative fares.

* **Don't Overwrite Silver Every Run**: Your original template did .mode("overwrite") with an empty table before the MERGE. If you do that, every new month will wipe out the previous months. We check if the table exists first.
----

1. `F.sha2(F.concat_ws(...), 256)`:
Takes the 6 columns that define a specific ride and hashes them into a single string (e.g.`, "a3f18b..."`). This acts exactly like your order_id.
2. `cleaned.limit(0).write...`:
Instead of writing a manual 20-line StructType schema with `StructField, limit(0)` grabs an empty DataFrame with the exact types inherited from Bronze and creates the table shell in one clean line.
3. `DeltaTable.isDeltaTable(...)`:
Prevents wiping out your Silver table on future runs when you process month `02`, `03`, etc.
</DETAILS>

#####DIAGNOSIS

differentiating real physical collisions (two different cabs doing similar trips) from logical duplicates (the same ride ingested twice or revised).

In [5]:

bronze_df = spark.read.format("delta").load(delta_bronze_table)

#define candidate keys
candidate_cols = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
]

#group by candidate key and count occurences

candidate_window = window.partitionBy(*candidate_cols)

collision_df = (
    bronze_df.withColumn("candidate_group_count",F.count("*").over(candidate_window))
    .filter(F.col("candidate_group_count")>1)
)

total_colliding_rows = collision_df.count()
distinct_colliding_groups = collision_df.select(candidate_cols).distinct().count()

print(f"{'='*60}\n COLLISION ANALYSIS REPORT: \n{candidate_cols}\n{'='*60}\n"
      f"Candidate Key Distinct Groups with > 1 row : {distinct_colliding_groups:,}\n"
      f"Total Rows involved in collisions          : {total_colliding_rows:,}\n"
      f"{'='*60}")

 COLLISION ANALYSIS REPORT: 
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']
Candidate Key Distinct Groups with > 1 row : 52,531
Total Rows involved in collisions          : 105,064


In [6]:
## examine why records with the exact same candidate key differ in non-key columns:
#list non key attributes to analyze differences
non_key_cols = [
    c for c in bronze_df.columns if c not in candidate_cols and c not in ["trip_id","quarantine_reason"]
]
print(f"analyzing variance across non-key coluumns: \n{non_key_cols}\n")

#order by candidate key so duplicate groups are displayed adjacent to each other

display_cols = candidate_cols + [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "congestion_surcharge",
    "total_amount",
    "payment_type"
]

#show the first 5 colliding groups

(
    collision_df.select(display_cols)
    .orderBy(*candidate_cols)
    .show(15,truncate=False)
)

analyzing variance across non-key coluumns: 
['passenger_count', 'RatecodeID', 'store_and_fwd_flag', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', '_source_file', '_ingest_at']

+--------+--------------------+---------------------+------------+------------+-------------+-----------+-----+-------+----------+------------+--------------------+------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|congestion_surcharge|total_amount|payment_type|
+--------+--------------------+---------------------+------------+------------+-------------+-----------+-----+-------+----------+------------+--------------------+------------+------------+
|2       |2025-01-01 00:00:07 |2025-01-01 00:06:31  |148         |45          |0.93         |-7.9       |-1.0 |-

In [7]:
# To programmatically know which columns are causing the discrepancy across all colliding groups without manually eyeballing rows:

# for each candidategroup, check if non-key coluumns have min != max (cvalues differ)

group_diff_exprs = [
    (F.min(col) != F.max(col)).cast("int").alias(f"diff_col_{col}")
    for col in non_key_cols
]

diff_summary = (
    collision_df.groupBy(*candidate_cols)
    .agg(*group_diff_exprs)
    .orderBy(*candidate_cols)
)

print("Summary of columns that DIFFER within colliding groups:")
print({'-'*55})
for col in non_key_cols:
  differing_groups = diff_summary.filter(F.col(f"diff_col_{col}")==1).count()
  pct = (differing_groups/distinct_colliding_groups)*100 if distinct_colliding_groups>0 else 0
  print(f"{col:<25} differes in {differing_groups:>5} groups ({pct:>5.1f}%)")

Summary of columns that DIFFER within colliding groups:
{'-------------------------------------------------------'}
passenger_count           differes in     0 groups (  0.0%)
RatecodeID                differes in     0 groups (  0.0%)
store_and_fwd_flag        differes in     1 groups (  0.0%)
payment_type              differes in  3816 groups (  7.3%)
fare_amount               differes in 52235 groups ( 99.4%)
extra                     differes in 25744 groups ( 49.0%)
mta_tax                   differes in 50279 groups ( 95.7%)
tip_amount                differes in   796 groups (  1.5%)
tolls_amount              differes in  4205 groups (  8.0%)
improvement_surcharge     differes in 52433 groups ( 99.8%)
total_amount              differes in 52531 groups (100.0%)
congestion_surcharge      differes in 42636 groups ( 81.2%)
Airport_fee               differes in  9193 groups ( 17.5%)
cbd_congestion_fee        differes in 29690 groups ( 56.5%)
_source_file              differes in     0 

In [8]:
#test the combined business key
candidate_combined = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
]
comb_window = window.partitionBy(*candidate_combined)
comb_collision_df = (
    bronze_df.withColumn("group_count",F.count("*").over(comb_window))
    .filter(F.col("group_count")>1)
)

coliding_groups = comb_collision_df.select(candidate_combined).distinct().count()
colliding_rows = comb_collision_df.count()

print(f"{'='*60}\n COMBINED KEY TEST:\n {candidate_combined}\n{'='*60}\n"
      f"colliding groups  :{coliding_groups:,}\n"
      f"colliding rows    :{colliding_rows:,}\n"
      f"{'='*60}")
#inspect 3 sample colliding groups side by side

print("\nInspecting colliding groups across surcharge columns:")

(
    comb_collision_df.select(
        "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "fare_amount", "trip_distance", "mta_tax",
        "congestion_surcharge", "total_amount", "payment_type"
    )
    .orderBy(*candidate_combined)
    .show(3,truncate=False)
)

 COMBINED KEY TEST:
 ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'fare_amount']
colliding groups  :296
colliding rows    :594

Inspecting colliding groups across surcharge columns:
+--------+--------------------+---------------------+-----------+-------------+-------+--------------------+------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|fare_amount|trip_distance|mta_tax|congestion_surcharge|total_amount|payment_type|
+--------+--------------------+---------------------+-----------+-------------+-------+--------------------+------------+------------+
|2       |2025-01-01 00:01:19 |2025-01-01 00:01:38  |0.0        |0.0          |0.0    |-2.5                |-2.5        |2           |
|2       |2025-01-01 00:01:19 |2025-01-01 00:01:38  |0.0        |0.0          |0.0    |2.5                 |2.5         |2           |
|2       |2025-01-01 00:43:53 |2025-01-01 00:44:19  |0.0        |0.0         

In [9]:
#  Verify: Are the Remaining 296 Groups ALL Offsetting Pairs?
# sum total_amount across each of the 296 colliding groups

pair_reconciliation = (
    comb_collision_df.groupBy(*candidate_combined)
    .agg(F.count("*").alias("records_in_groups"),
         F.round(F.sum("total_amount"),2).alias("net_group_amount"),
         F.min("total_amount").alias("min_amount"),
         F.max("total_amount").alias("max_amount")
    )
)

exact_cancellaing_pairs = pair_reconciliation.filter(
    F.col("net_group_amount")==0.0).count()

print(
    f"Total colliding groups                  :{coliding_groups}\n"
    f"Groups that cancel out to exactly $0.00 :{exact_cancellaing_pairs}\n"
    f"\n\nSample of net amounts across groups:\n"
)
pair_reconciliation.orderBy(F.desc("records_in_groups")).show(10, truncate=False)

Total colliding groups                  :296
Groups that cancel out to exactly $0.00 :294


Sample of net amounts across groups:

+--------+--------------------+---------------------+------------+------------+-------------+-----------+-----------------+----------------+----------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|records_in_groups|net_group_amount|min_amount|max_amount|
+--------+--------------------+---------------------+------------+------------+-------------+-----------+-----------------+----------------+----------+----------+
|2       |2025-01-15 10:08:42 |2025-01-15 10:13:06  |48          |50          |1.06         |0.0        |3                |4.75            |-4.0      |4.75      |
|2       |2025-01-22 07:35:30 |2025-01-22 07:46:44  |100         |158         |1.57         |0.0        |3                |4.75            |-4.0      |4.75      |
|2       |2025-01-01 00:01:19 |2025-01-01 00:01:38  |16

<details>
<summary>VERDICT</summary>  
294 out of 296 (99.3%) are Exact $0.00 Accounting Offsets

This confirms the root cause with certainty.   
- They are NOT duplicate rides.   
- They are NOT physical cab collisions.   
- They are Meter Voids and Reversals.  

1. In **294 out of 296 groups** ,  the net revenue is `$0.00`. The driver started the meter, surcharges were added `($1.50, $2.50, $4.00, $5.75)`, the trip was aborted, and the **meter software generated an exact offsetting credit** with identical timestamps and 0.0 base fare.   
And in the other 2 groups? A ride was charged, voided, and rebilled with a corrected surcharge (+4.75 vs -4.00).

----
2. Instead of calling those 294 records duplicates, we give them meaningful accounting categories:
- `NEGATIVE_OR_REVERSAL_TRANSACTION`: For rows where `fare_amount < 0` or `total_amount < 0`.
- `METER_ABORT_ZERO_DISTANCE`: For rows where a meter was opened and immediately closed (`trip_distance == 0.0 `and `fare_amount == 0.0`) with no passenger ride.
- `DUPLICATE_RECORD`: Reserved strictly for records that match on all business dimensions including `total_amount`.

</details>

## Step 2.1 REMEDIATED SILVER PIPELINE

In [11]:
#1. Paths
delta_bronze_table = os.path.join(BRONZE, "yellow_taxi_delta")
delta_silver_table = os.path.join(SILVER, "yellow_taxi_delta")
delta_quarantine_table = os.path.join(SILVER,"yellow_taxi_quarantine_delta")

#2. Read Bronze
bronze_df = spark.read.format("delta").load(delta_bronze_table)
all_cols = bronze_df.columns

#3. generate validated Surrogate Business Key (which includes trip_distance & total distance)
keyed_df = (bronze_df
  # Candidate Business Key
            .withColumn(
                "trip_id",
                F.sha2(F.concat_ws(
                    "||",
                    F.coalesce(F.col("VendorID").cast("string"),F.lit("")),
                    F.coalesce(F.col("tpep_pickup_datetime").cast("string"),F.lit("")),
                    F.coalesce(F.col("tpep_dropoff_datetime").cast("string"),F.lit("")),
                    F.coalesce(F.col("PULocationID").cast("string"),F.lit("")),
                    F.coalesce(F.col("DOLocationID").cast("string"),F.lit("")),
                    F.coalesce(F.col("trip_distance").cast("string"),F.lit("")),
                    F.coalesce(F.col("fare_amount").cast("string"),F.lit("")),
                    F.coalesce(F.col("total_amount").cast("string"),F.lit("")),
                    F.coalesce(F.col("payment_type").cast("string"),F.lit(""))
                    ),256
                       )
                )
            .withColumn(
                "_record_fingerprint",
                F.sha2(
                    F.concat_ws(
                        "||",
                        *[F.coalesce(F.col(c).cast("string"),F.lit("__NULL__"))for  c in sorted(all_cols)]
                    ),256
                )
            ).withColumn(
                "_completeness_score",
                sum(F.when(F.col(c).isNotNull(),1).otherwise(0) for c in all_cols)
            )
          )

#4.  PHASE 1: Independent quality validation (Decoupled from deduplication)
# collect all violations into an array so no rule is shadowed

violated_df = keyed_df.withColumn(
    "quality_errors",
    F.array_compact(
        F.array(
            F.when(F.col("tpep_pickup_datetime").isNull(), F.lit("NULL_PICKUP_DATETIME"))
            .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"), F.lit("DROPOFF_BEFORE_PICKUP"))
            .when(F.col("trip_distance") < 0, F.lit("NEGATIVE_DISTANCE"))
            .when((F.col("fare_amount") < 0) | (F.col("total_amount")<0), F.lit("NEGATIVE_OR_REVERSAL_TRANSACTION"))
            .when((F.col("trip_distance") == 0.0) & (F.col("fare_amount") == 0.0), F.lit("METER_ABORT_ZERO_DISTANCE"))
            .when(
                  (F.year("tpep_pickup_datetime") != int(year)) |
                  (F.month("tpep_pickup_datetime") != int(month)),
                    F.lit("PICKUP_DATETIME_NOT_IN_SPECIFIED_YEAR_MONTH")
        )
    )
)
    ).withColumn("is_valid", F.size(F.col("quality_errors"))==0)

  #5. PHASE 2: Deterministic Deduplication
  #order Priority:
  #1. Clean records over dirty records
  #2. Records with highest completeness (least NULLS)
  #3. Stable mathematical fingerprint across all columns


dedup_window = (window.partitionBy("trip_id")
  .orderBy(F.col("is_valid").desc(),
           F.col("_completeness_score").desc(),
           F.col("_record_fingerprint").asc()
           )
  )

deduped_df = (
    violated_df
    .withColumn("duplicate_rank",F.row_number().over(dedup_window))
    .withColumn("quarantine_reasons",
                F.when(
                    F.col("duplicate_rank")>1,
                    F.array_union(F.col("quality_errors"), F.array(F.lit("DUPLICATE_RECORD"))
                )).otherwise(F.col("quality_errors"))
    ).drop("_completeness_score","quuality_errors")
  )

#6. Split Streams
#Silver muust be 100% valid AND the primary winning record
valid_df = (
    deduped_df
    .filter((F.col("is_valid")==True) & (F.col("duplicate_rank")==1))
    .drop("quarantine_reasons","duplicate_rank","is_valid","_record_fingerprint")
)

#Quarantine: Any record that failed quality checks OR was a duplicate
# Define explicit Batch & Source Metadata

batch_id = f"yellow_{year}_{month}"
source_file = f"yellow_tripdata_{year}-{month}.parquet"


# Enrich Quarantine records with audit lineage & composite key
quarantine_enriched_df =(
    deduped_df
    .filter((F.size(F.col("quarantine_reasons"))>0))
    .withColumn("quarantine_reason", F.array_join("quarantine_reasons","; "))
    # add audit metadata
    .withColumn("_batch_id",F.lit(batch_id))
    .withColumn("_source_file",F.lit(source_file))
    .withColumn("_quarantined_at",F.current_timestamp())
    #composite stable identifier
    .withColumn("_quarantine_id",
                F.sha2(F.concat_ws("||",
                                   F.col("_record_fingerprint"), F.col("_batch_id")),256)
                ).drop("duplicate_rank","is_valid","quarantine_reasons")
    )

#7. Write to Silver (Fast direct overwrite on rebuild, or merge on incremental)
print("Writing validated passenger trips to Silver...")
valid_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(delta_silver_table)

#8. Write to Quarantine (Overwrite on rebuild so rerun doesn't duplicate logs)
print("Writing invalid/aborted trips to Quarantine...")

# Indempotent Quarantine write via Batch Replacement (Replacewhere)
# If table doesnt exist, create it ; if it exists, atomically replace ONLY this batch

table_exists = DeltaTable.isDeltaTable(spark,delta_quarantine_table)
schema_is_outdated = False

if table_exists:
  existing_cols = spark.read.format("delta").load(delta_quarantine_table).columns
  # if the on disk table is from an older run missing the new metadata columns:
  if "_batch_id" not in  existing_cols:
    schema_is_outdated = True

if not table_exists or schema_is_outdated:
    print("Initialising Quarantine table with new partitioned schema...")
    (
        quarantine_enriched_df
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema","true")
        .partitionBy("_batch_id")
        .save(delta_quarantine_table)
    )

else:
  print(f"Quarantine table schema verified. Idempotently replacing batch '{batch_id}'...")
  (
    quarantine_enriched_df
    .write.format("delta")
    .mode("overwrite")
    .option("replacewhere", f"_batch_id = '{batch_id}'")
    .save(delta_quarantine_table)
  )

#9. Ingest Summary
silver_count = spark.read.format("delta").load(delta_silver_table).count()
quarantine_count = (
    spark.read.format("delta").load(delta_quarantine_table)
    .filter(F.col("_batch_id")==batch_id)
    .count()
)


print(
    f"{'-'*5} INGESTION REPORT {'-'*5}\n"
    f"Rows in Silver (CLEAN):     {silver_count:,}\n"
    f"Rows in Quarantine:         {quarantine_count:,}\n"
    f"Total accounted rows:       {(silver_count+quarantine_count):,}\n"
    f"Bronze raw inflow records:  {bronze_df.count():,}"
)



Writing validated passenger trips to Silver...
Writing invalid/aborted trips to Quarantine...
Initialising Quarantine table with new partitioned schema...
----- INGESTION REPORT -----
Rows in Silver (CLEAN):     3,330,162
Rows in Quarantine:         145,064
Total accounted rows:       3,475,226
Bronze raw inflow records:  3,475,226


<details><summary>Alternative: Idempotent Write via Delta MERGE</summary>

If architecture mandates row-level upserts rather than batch replacements, here is the equivalent programmatic Delta `MERGE` implementation:

```
# Initialize empty table if missing
if not DeltaTable.isDeltaTable(spark, delta_quarantine_table):
    (
        quarantine_enriched_df
        .limit(0)
        .write
        .format("delta")
        .mode("overwrite")
        .save(delta_quarantine_table)
    )

quarantine_target = DeltaTable.forPath(spark, delta_quarantine_table)

(
    quarantine_target.alias("target")
    .merge(
        quarantine_enriched_df.alias("source"),
        "target._quarantine_id = source._quarantine_id"
    )
    .whenMatchedUpdate(set={
        "_quarantined_at": "source._quarantined_at",
        "quarantine_reason": "source.quarantine_reason"
    })
    .whenNotMatchedInsertAll()
    .execute()
)
```

</details>

In [12]:
## checking if bad data is properly classified

(
    spark.read.format("delta").load(delta_quarantine_table)
    .groupBy("quarantine_reason")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.sum("total_amount"),2).alias("subtotal_revenue")
    )
    .orderBy("record_count")
    .show(truncate =False)
)

+-------------------------------------------+------------+----------------+
|quarantine_reason                          |record_count|subtotal_revenue|
+-------------------------------------------+------------+----------------+
|PICKUP_DATETIME_NOT_IN_SPECIFIED_YEAR_MONTH|22          |611.3           |
|DROPOFF_BEFORE_PICKUP                      |124         |3109.63         |
|METER_ABORT_ZERO_DISTANCE                  |479         |894.42          |
|NEGATIVE_OR_REVERSAL_TRANSACTION           |144439      |-1331293.3      |
+-------------------------------------------+------------+----------------+



In [13]:
## to prove that deduplication is fully deterministic.
# It processes the exact same raw data twice under wildly different partition arrangements
# (2 partitions vs 16 partitions) and checks if the retained Silver dataset is identical down to the byte.
def process_silver_pipeline(input_df):
    """Encapsulates the deterministic ranking logic."""
    k_df = (
        input_df
        .withColumn(
            "trip_id",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("VendorID").cast("string"), F.lit("")),
                    F.coalesce(F.col("tpep_pickup_datetime").cast("string"), F.lit("")),
                    F.coalesce(F.col("tpep_dropoff_datetime").cast("string"), F.lit("")),
                    F.coalesce(F.col("PULocationID").cast("string"), F.lit("")),
                    F.coalesce(F.col("DOLocationID").cast("string"), F.lit("")),
                    F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
                    F.coalesce(F.col("fare_amount").cast("string"), F.lit("")),
                    F.coalesce(F.col("total_amount").cast("string"), F.lit("")),
                    F.coalesce(F.col("payment_type").cast("string"), F.lit(""))
                ), 256
            )
        )
        .withColumn(
            "full_row_hash",
            F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("__NULL__")) for c in sorted(all_cols)]), 256)
        )
        .withColumn("completeness_score", sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in all_cols))
        .withColumn(
            "is_valid",
            (F.col("tpep_pickup_datetime").isNotNull()) &
            (F.col("tpep_dropoff_datetime") >= F.col("tpep_pickup_datetime")) &
            (F.col("trip_distance") >= 0) &
            (F.col("fare_amount") >= 0) &
            (F.col("total_amount") >= 0) &
            ~((F.col("trip_distance") == 0.0) & (F.col("fare_amount") == 0.0)) &
            (F.year("tpep_pickup_datetime") == int(year)) &
            (F.month("tpep_pickup_datetime") == int(month))
        )
    )

    win = window.partitionBy("trip_id").orderBy(
        F.col("is_valid").desc(),
        F.col("completeness_score").desc(),
        F.col("full_row_hash").asc()
    )

    return (
        k_df
        .withColumn("rank", F.row_number().over(win))
        .filter((F.col("is_valid") == True) & (F.col("rank") == 1))
        .select("trip_id", "total_amount", "full_row_hash")
    )

print("Running Determinism Acceptance Test across different partition topologies...")

# Run 1: Low-parallelism arrangement (2 partitions)
run_1 = process_silver_pipeline(bronze_df.repartition(2))

# Run 2: High-parallelism arrangement (16 partitions with random shuffle)
run_2 = process_silver_pipeline(bronze_df.repartition(16))

# Verify bit-for-bit equivalence
diff_1 = run_1.subtract(run_2).count()
diff_2 = run_2.subtract(run_1).count()

print(f"Discrepancies (Run 1 minus Run 2): {diff_1}")
print(f"Discrepancies (Run 2 minus Run 1): {diff_2}")

assert diff_1 == 0 and diff_2 == 0, "DETERMINISM FAILED: Different records were retained across runs!"
print("\n✅ ACCEPTANCE TEST PASSED: Deduplication is 100% deterministic regardless of physical partition layout.")

Running Determinism Acceptance Test across different partition topologies...
Discrepancies (Run 1 minus Run 2): 0
Discrepancies (Run 2 minus Run 1): 0

✅ ACCEPTANCE TEST PASSED: Deduplication is 100% deterministic regardless of physical partition layout.


**treating Silver as ACID/idempotent while treating Quarantine as a naive append log**.   
 If an orchestrator (like Airflow) retries a failed task, or if you backfill a month, `.mode("append")` silently duplicates every quarantined record. Your reject counts, error metrics, and financial reconciliation will artificially inflate on every rerun.    
- We implement `batch-aware metadata tracking` and an `idempotent write pattern` using a stable quarantine identifier (`_quarantine_id`) and Delta Lake's `atomic batch replacement` (or Delta `MERGE`).

<details><summary>The Architectural Fix</summary>
Every quarantined record must be enriched with five audit columns before writing:   

Column_________________________Type_________________________Purpose      
- `_record_fingerprint`____`STRING`____SHA-256 hash of all raw column values (stable content ID)  
- `_batch_id`____`STRING`____Ingestion batch identifier (e.g., yellow_2024_01)  
- `_source_file`____ `STRING`____ Source filename (e.g., yellow_tripdata_2024-01.parquet)   
- `_quarantined_at`____ `TIMESTAMP`____ 	Ingestion audit timestamp (current_timestamp())   
- `quarantine_reason`____ `STRING`____ Semicolon-delimited failure reasons
- `_quarantine_id`____ `STRING`____ Composite key: sha2(_record_fingerprint || _batch_id)    

For idempotent writes, the industry-standard batch pattern in Delta Lake is atomic partition/predicate replacement (replaceWhere) or a Delta MERGE on _quarantine_id.

We will use replaceWhere scoped to _batch_id (with Delta MERGE fallback):

- **Run 1**: Ingests batch yellow_2024_01 (144,564 records written).
- **Run 2** (Rerun): Atomically replaces only the records for yellow_2024_01. Total count remains exactly 144,564.
- **Run 3** (Rerun): Replaces again. Total count remains exactly 144,564.
- **Batch yellow_2024_02**: Appends new batch alongside 2024_01 without interference.
</details>
<details><summary>Please Note</summary>

- Why `replaceWhere` over basic `MERGE` for batch quarantine?   
A Delta `MERGE` on  144,000  rows across a full table **requires an expensive shuffle join**. In contrast, `replaceWhere` on` _batch_id` takes advantage of Delta Lake's ACID transaction log: it **performs an instant metadata file swap** for that specific batch **in seconds** with zero risk of row multiplication.
</details>


In [14]:
"""
to prove rerun idempotency.
It simulates three consecutive pipeline executions on the same batch and
verifies that the row count does not increase:
"""
print(f"""
    {'='*60}
    {'Starting acceptance test: Quarantine rerun Idepotency':^60}
    {'='*60}
    """)
test_batch_id = f"yellow_{year}_{month}"

def execute_quarantine_write(iteration: int):
  """Simulates an execution of the quarantine write step"""
  (
      quarantine_enriched_df
      .write.format("delta")
      .mode("overwrite")
      .option("replaceWhere", f"_batch_id = '{test_batch_id}'")
      .save(delta_quarantine_table)
  )
  total_rows = spark.read.format("delta").load(delta_quarantine_table).count()
  batch_rows = (
      spark.read.format("delta").load(delta_quarantine_table)
      .filter(F.col("_batch_id")==test_batch_id)
      .count()
  )
  distinct_ids = (
      spark.read.format("delta").load(delta_quarantine_table)
      .filter(F.col("_batch_id")== test_batch_id)
      .select("_quarantine_id")
      .distinct()
      .count()
  )
  print(f"""
  Iteration {iteration} -> Total table rows: {total_rows:,}
                           Batch rows:       {batch_rows:,}
                           Unique IDs:       {distinct_ids:,}
  """)
  return batch_rows, distinct_ids

# Run1 : first ingestion
count_1, distinct_1 = execute_quarantine_write(1)

# Run2 : rerun
count_2, distinct_2 = execute_quarantine_write(2)

# Run3 : rerun
count_3, distinct_3 = execute_quarantine_write(3)

print(f"""
{'-'*60}
Run 1 Count: {count_1:,}
Run 2 Count: {count_2:,}
Run 3 Count: {count_3:,}
""")

#invariance Assertions
assert count_1 == count_2 == count_3 ,(
    f"INDEPOTENCY FAILURE: Row count grew across runs: ({count_1} -> {count_2} -> {count_3})"
)

assert distinct_1 == distinct_2 == distinct_3 ,(
    f"INTEGRITY FAILURE: Duplicate _quarantine_id records detected in Quarantine"
)
print("Acceptance test passed: Quarantine is 100% idempotent across repeated executions")



       Starting acceptance test: Quarantine rerun Idepotency    
    

  Iteration 1 -> Total table rows: 145,064
                           Batch rows:       145,064
                           Unique IDs:       145,064
  

  Iteration 2 -> Total table rows: 145,064
                           Batch rows:       145,064
                           Unique IDs:       145,064
  

  Iteration 3 -> Total table rows: 145,064
                           Batch rows:       145,064
                           Unique IDs:       145,064
  

------------------------------------------------------------
Run 1 Count: 145,064
Run 2 Count: 145,064
Run 3 Count: 145,064      

Acceptance test passed: Quarantine is 100% idempotent across repeated executions


## Step 3: Gold (aggregates)
builds the gold aggregates, the business ready layer

In [15]:
#1. define paths
delta_silver_table = os.path.join(SILVER, "yellow_taxi_delta")
delta_gold_table = os.path.join(GOLD,"daily_taxi_summary")

#2. Read the silver delta table
silver_df = spark.read.format("delta").load(delta_silver_table)

#3. Aggregate into Business metrics(the gold transform)
gold_df = (
    silver_df.withColumn("pickup_date",F.to_date("tpep_pickup_datetime"))       #derive pickup date from timestamp
    .groupBy("pickup_date","VendorID")                                      #group by date and vendor
    .agg(                                                                   #calculate executive KPIs
        F.count("*").alias("total_trips"),
        F.sum("passenger_count").alias("total_passengers"),
        F.round(F.sum("total_amount"),2).alias("total_revenue"),
        F.round(F.sum("fare_amount"),2).alias("total_fare"),
        F.round(F.sum("tip_amount"),2).alias("total_tips"),
        F.round(F.avg("trip_distance"),2).alias("avg_distance_miles"),
        F.round(F.avg("fare_amount"),2).alias("avg_fare_per_trip")

)
 .orderBy("pickup_date","VendorID")
)

#4. Save to Gold
#----NB can save as:
#                     "delta" -> recommended  for data lakes
# or
#                     "Parquet" -> for external tools
gold_df.write.format("delta").mode("overwrite").save(delta_gold_table)
print(f"Gold layer written successfully")

#5. preview the dashboard_ready table

print("\n ---GOLD SUMMARY PREVIEW---")
spark.read.format("delta").load(delta_gold_table).show(10,truncate=False)

Gold layer written successfully

 ---GOLD SUMMARY PREVIEW---
+-----------+--------+-----------+----------------+-------------+----------+----------+------------------+-----------------+
|pickup_date|VendorID|total_trips|total_passengers|total_revenue|total_fare|total_tips|avg_distance_miles|avg_fare_per_trip|
+-----------+--------+-----------+----------------+-------------+----------+----------+------------------+-----------------+
|2025-01-01 |1       |17359      |19603           |482062.31    |348605.74 |48918.89  |3.55              |20.08            |
|2025-01-01 |2       |65034      |91359           |1892554.68   |1339599.11|213475.58 |5.07              |20.6             |
|2025-01-01 |6       |13         |NULL            |624.11       |44.95     |0.0       |15.79             |3.46             |
|2025-01-01 |7       |15         |23              |320.15       |185.0     |55.15     |1.71              |12.33            |
|2025-01-02 |1       |18685      |21433           |515580.37    

 The most common business need is a **Daily Performance** & **Revenue Summary** :
 - tracking total revenue
 - trip volume
 - tips
 - average distance per day.    

##### observations
1. Specific Table Path (daily_taxi_summary):   
- In production, you never write directly to the root GOLD folder because your project will eventually have multiple Gold tables (e.g., ***daily_summary***, ***airport_pickups***, ***driver_tip_analysis***). Always create a named subfolder.
2. Metric Rounding (F.round(..., 2)):   
- Financial and business metrics (***total_revenue***, ***avg_fare***) should always be rounded to 2 decimal places to avoid messy floating-point outputs like `$14.20000000004` on dashboards.
3. Delta vs. Parquet for Gold:   
    - **When to use Delta**: If internal teams query Gold using Spark, Trino, or Databricks, use Delta to retain ACID transactions and time travel.
    - **When to use Parquet**: If you need to hand these files directly to older tools or lightweight Python libraries that do not read Delta tables natively.


## Step 4: Check the numbers (Data reconciliation & pipeline auditing )
compare row counts and totals across bronze -> silver -> gold

In [16]:
#1. Define paths matching the ebtire pipeline
delta_bronze_table = os.path.join(BRONZE, "yellow_taxi_delta")
delta_silver_table = os.path.join(SILVER, "yellow_taxi_delta")
delta_quarantine_table = os.path.join(SILVER,"yellow_taxi_quarantine_delta")
delta_gold_table = os.path.join(GOLD,"daily_taxi_summary")

#2. extract bronze metrics
bronze_df = spark.read.format("delta").load(delta_bronze_table)
bronze_count = bronze_df.count()
bronze_total = bronze_df.agg(F.sum("total_amount")).collect()[0][0] or 0.0

#3. Extract silver metrics (Row count + Total revenue)
silver_df = spark.read.format("delta").load(delta_silver_table)
silver_count = silver_df.count()
silver_total = silver_df.agg(F.sum("total_amount")).collect()[0][0] or 0.0

#3.1 Extract Quarantine metrics(Dirty/Rejected volume & revenue)
if DeltaTable.isDeltaTable(spark, delta_quarantine_table):
    quarantine_df = spark.read.format("delta").load(delta_quarantine_table)
    quarantine_count = quarantine_df.count()
    quarantine_total = quarantine_df.agg(F.sum("total_amount")).collect()[0][0] or 0.0
else:
    quarantine_count = 0
    quarantine_total = 0.0

#4. Extract Gold metrics (aggregated rows + Toatal revenue)
gold_df = spark.read.format("delta").load(delta_gold_table)
gold_count = gold_df.count()
gold_total = gold_df.agg(F.sum("total_revenue")).collect()[0][0] or 0.0

#5. Calculations and Reconciliations
accounted_rows = silver_count + quarantine_count


accounted_rev = silver_total + quarantine_total
var_bronze_vs_accounted = abs(bronze_total-accounted_rev)
var_silver_vs_gold = abs(silver_total-gold_total)

#6. Print executive Audit report
print(f"{'='*55}\n"
        f"{'Medallion PIPELINE AUTIT REPORT':^55} \n"
        f"{'='*55}\n"
        f"{'-'*15} RAW COUNT INTEGRITY {'-'*15}\n"
        f"Bronze raw inflow rows:                 {bronze_count:,}\n"
        f"Silver Cleaned rows:                    {silver_count:,}\n"
        f"Quarantined (bad) Rows:                 {quarantine_count:,}\n"
        f"Gold aggregate rows:                    {gold_count:,}\n"
        f"\n{'-'*15} FINANCIAL RECONCILIATION {'-'*15}\n"
        f"Bronze raw inflaw revenue:              ${bronze_total:,.2f}\n"
        f"Silver cleaned revenue:                 ${silver_total:,.2f}\n"
        f"Quarantined bad revenue:                ${quarantine_total:,.2f}\n"
        f"Total ACCOUNTED revenue(silver+quarn):  ${accounted_rev:,.2f}\n"
        f"Gold kpi Summary revenue:               ${gold_total:,.2f}\n"
        f"{'-'*55}\n"
        f"Variance (Bronze vs Accounted):         ${var_bronze_vs_accounted:,.2f}\n"
        f"Variance (Silver vs Gold):              ${var_silver_vs_gold:,.2f}\n"
        f"{'='*55}"
        )

#7. Automated Quality Gates (data integrity Assertions)

assert bronze_count > 0, "Audit failed: Bronze layer is empty"
assert silver_count > 0, "Audit failed: Silver layer is empty"
assert silver_count <= bronze_count,  "Audit failed: Silver has more rows than Bronze"
assert gold_count < silver_count, "Audit failed: Gold did not aggreagte data properly"

#8. Financial reconciliation assertions:
#Gate A: allow upto a $1.00 variance due to floating point rounding across the rows(Gold MUST match silver)
assert var_silver_vs_gold < 1.0, (
    f"Audit failed: Financial mismatch! Silver=${silver_total:,.2f} vs Gold=${gold_total:,.2f}"
    )
#Gate B: 100% of Bronze revenue is accouunted for in either Silver or Quarantine (ignoring duplicate records)
assert var_bronze_vs_accounted < 5.0, (
    f"Audit failed: unaccounted revenue! Bronze=${bronze_total:,.2f} vs Accounted=${accounted_rev:,.2f}"
)
print("\n ALL CHECKS PASSED: PIPELINE IS RECONCILED AND HEALTHY")

            Medallion PIPELINE AUTIT REPORT             
--------------- RAW COUNT INTEGRITY ---------------
Bronze raw inflow rows:                 3,475,226
Silver Cleaned rows:                    3,330,162
Quarantined (bad) Rows:                 145,064
Gold aggregate rows:                    124

--------------- FINANCIAL RECONCILIATION ---------------
Bronze raw inflaw revenue:              $89,005,026.80
Silver cleaned revenue:                 $90,331,704.75
Quarantined bad revenue:                $-1,326,677.95
Total ACCOUNTED revenue(silver+quarn):  $89,005,026.80
Gold kpi Summary revenue:               $90,331,704.75
-------------------------------------------------------
Variance (Bronze vs Accounted):         $0.00
Variance (Silver vs Gold):              $0.00

 ALL CHECKS PASSED: PIPELINE IS RECONCILED AND HEALTHY


In [17]:
# Check total amount of all negative/invalid fares in Bronze
bad_bronze_money = bronze_df.filter(F.col("fare_amount") < 0).agg(F.sum("total_amount")).collect()[0][0]
print(f"Total Negative Fares in Bronze: ${bad_bronze_money:,.2f}")

Total Negative Fares in Bronze: $-1,330,081.05


Data has millions of rows that change month-to-month, you **cannot** hardcode static numbers like `assert bronze_count == 5` . Instead, you write ***relational assertions***:

1. `Silver rows <= Bronze rows`: Because deduplication and quality filters remove dirty rows.
2. `Gold Revenue == Silver Revenue`: The total money in the executive dashboard (Gold) ***MUST balance*** with the money in the cleaned records (Silver).

#### This gives you:
 - **Financial Integrity**: prove that aggregation logic didn't accidentally discard money or double-count payments between Silver and Gold.
- **Data Drop Visibility**: It prints the exact percentage of corrupted/duplicate rides cleaned out between Bronze and Silver.
- **Automated Pipeline Gate**: If this runs inside an orchestrator (like Apache Airflow or Databricks Workflows), any AssertionError will immediately fail the pipeline and ***alert on Slack/PagerDuty*** before dirty data reaches dashboards.



### Step 5  : Serve (Gold layer goes to the consumer)  

reads gold table with a serving engine - DuckDB OVER THE dELTA TABLE AND PREVIEWS + totals the revenue

serving is the handoff from producer(spark) to consumer (analyst, dashboardm downstream service).   

in paid  world, this is where you point a warehouse (snowflake) at the gold layer.   
Free version: serve straight from the Delta table with DuckDB - no warehouse needed

In [18]:
try:
  import duckdb
except ImportError:
  %pip install -q duckdb

gold_cron = duckdb.connect()

preview = gold_cron.execute(
    f"SELECT * FROM delta_scan('{delta_gold_table}') limit 10;"
).fetchdf()
print(preview)

served_total = gold_cron.execute(
    f"SELECT SUM(total_revenue) FROM delta_scan('{delta_gold_table}')"
).fetchdf().squeeze()
print(f"Served gold revenue: ${served_total:,.2f}")

  pickup_date  VendorID  total_trips  total_passengers  total_revenue  \
0  2025-01-01         1        17359             19603      482062.31   
1  2025-01-01         2        65034             91359     1892554.68   
2  2025-01-01         6           13              <NA>         624.11   
3  2025-01-01         7           15                23         320.15   
4  2025-01-02         1        18685             21433      515580.37   
5  2025-01-02         2        63340             92375     1907243.07   
6  2025-01-02         6           13              <NA>         377.60   
7  2025-01-02         7           18                24         369.52   
8  2025-01-03         1        20201             23533      541227.39   
9  2025-01-03         2        68361             98799     1952417.14   

   total_fare  total_tips  avg_distance_miles  avg_fare_per_trip  
0   348605.74    48918.89                3.55              20.08  
1  1339599.11   213475.58                5.07              20.